# TP3 – Données Gold et format Parquet

Ryma Dinari – Retail Intelligence Platform

Étapes 5, 7 et 8 de l'UC-06 : je fais les jointures et les agrégations à partir des données Silver, j'enregistre les résultats dans la zone Gold en Parquet, puis je compare Parquet avec le format texte d'origine.

Je réponds à deux questions du jour 1 :
- **Q1 / Q5** : comment évolue le chiffre d'affaires, et quels magasins marchent le mieux ? → `ventes_magasin_mois`
- **Q3** : quelles catégories de produits vendent le plus ? → `ventes_categorie_annee`

In [1]:
import sys, time
sys.path.insert(0, "..")

from config import *
from pyspark.sql import functions as F

spark = creer_session()

ventes = spark.read.parquet(str(SILVER / "sales"))
produits = spark.read.parquet(str(SILVER / "products"))
magasins = spark.read.parquet(str(SILVER / "stores"))
dates = spark.read.parquet(str(SILVER / "dates"))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/18 14:53:59 WARN Utils: Your hostname, rymadinari-VivoBook-ASUSLaptop-M1702QA-M1702QA, resolves to a loopback address: 127.0.1.1; using 10.188.47.136 instead (on interface wlp1s0)
26/11/18 14:53:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/18 14:53:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/11/18 14:54:00 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## 1. Jointures

J'ai vérifié dans le notebook 01 que les clés existent toujours dans les tables de référence (0 orphelin) et qu'elles sont uniques de leur côté.
C'est une relation plusieurs-à-un : une vente a un seul produit, un seul magasin, une seule date. Donc après une jointure, le nombre de lignes ne doit pas changer.
Je fais quand même des jointures `left` pour ne perdre aucune vente si un jour une clé manque.

In [2]:
ventes_enrichies = (
    ventes
    .join(dates.select("d_date_sk", "d_date", "d_year", "d_moy"),
          ventes.ss_sold_date_sk == dates.d_date_sk, "left")
    .join(magasins.select("s_store_sk", "s_store_name", "s_state"),
          ventes.ss_store_sk == magasins.s_store_sk, "left")
    .join(produits.select("i_item_sk", "i_category"),
          ventes.ss_item_sk == produits.i_item_sk, "left")
)

print("lignes avant :", ventes.count())
print("lignes après :", ventes_enrichies.count())
print("ventes sans date     :", ventes_enrichies.filter("d_date is null").count())
print("ventes sans magasin  :", ventes_enrichies.filter("s_store_name is null").count())
print("ventes sans catégorie:", ventes_enrichies.filter("i_category is null").count())

lignes avant : 667579


lignes après : 667579


ventes sans date     : 0


ventes sans magasin  : 0


ventes sans catégorie: 0


Même nombre de lignes avant et après, et aucune vente sans correspondance : les jointures sont correctes.

## 2. Gold 1 – ventes par magasin et par mois

- **Granularité** : une ligne = un magasin pour un mois d'une année.
- **CA** = somme de `ss_net_paid` (montant payé par le client après remises et coupons, hors taxes).
- **Nombre de tickets** = nombre de `ss_ticket_number` différents (un ticket = un passage en caisse, vérifié dans le notebook 01).
- **Panier moyen** = CA / nombre de tickets.

In [3]:
ventes_magasin_mois = (
    ventes_enrichies
    .groupBy("s_store_sk", "s_store_name", "s_state", "d_year", "d_moy")
    .agg(
        F.sum("ss_net_paid").alias("ca"),
        F.sum("ss_quantity").alias("quantite"),
        F.countDistinct("ss_ticket_number").alias("nb_tickets"),
        F.count("*").alias("nb_lignes"),
        F.sum("ss_net_profit").alias("marge"),
    )
    .withColumn("panier_moyen", F.round(F.col("ca") / F.col("nb_tickets"), 2))
    .withColumnRenamed("d_year", "annee")
    .withColumnRenamed("d_moy", "mois")
    .orderBy("s_store_sk", "annee", "mois")
)
ventes_magasin_mois.show(10)
print("nombre de lignes :", ventes_magasin_mois.count())

+----------+------------+-------+-----+----+----------+--------+----------+---------+----------+------------+
|s_store_sk|s_store_name|s_state|annee|mois|        ca|quantite|nb_tickets|nb_lignes|     marge|panier_moyen|
+----------+------------+-------+-----+----+----------+--------+----------+---------+----------+------------+
|         0|      Forest|     CO| 2001|   1|1386332.96|   44609|       111|      861|-846846.06|    12489.49|
|         0|      Forest|     CO| 2001|   2|1538610.08|   47166|       123|      917|-781128.39|    12509.03|
|         0|      Forest|     CO| 2001|   3|1850977.23|   50304|       125|     1015|-717236.45|    14807.82|
|         0|      Forest|     CO| 2001|   4|1733270.60|   49850|       132|      979|-854730.81|    13130.84|
|         0|      Forest|     CO| 2001|   5|1574749.95|   47860|       120|      926|-743690.06|    13122.92|
|         0|      Forest|     CO| 2001|   6|1423436.15|   40140|       109|      773|-561119.58|    13059.05|
|         

nombre de lignes : 720


720 lignes = 12 magasins × 60 mois (2001 à 2005), donc chaque magasin a des ventes tous les mois.
Le panier moyen est d'environ 13 000 € : un ticket contient en moyenne 7 à 8 lignes avec des quantités jusqu'à 100. Ce n'est pas réaliste, ça vient du générateur.

In [4]:
# vérification : le total du Gold doit être égal au total du Silver
total_silver = ventes.agg(F.sum("ss_net_paid")).first()[0]
total_gold = ventes_magasin_mois.agg(F.sum("ca")).first()[0]
print("CA total Silver :", total_silver)
print("CA total Gold   :", total_gold)

CA total Silver : 1150529586.37
CA total Gold   : 1150529586.37


Les deux totaux sont identiques : l'agrégation n'a ni perdu ni compté deux fois des ventes.

In [5]:
# exemple d'utilisation : CA par année, tous magasins confondus (Q1)
ventes_magasin_mois.groupBy("annee").agg(F.sum("ca").alias("ca"), F.countDistinct("mois").alias("nb_mois")) \
                   .orderBy("annee").show()

+-----+------------+-------+
|annee|          ca|nb_mois|
+-----+------------+-------+
| 2001|232498659.83|     12|
| 2002|230947265.82|     12|
| 2003|233575185.63|     12|
| 2004|234344934.29|     12|
| 2005|219163540.80|     12|
+-----+------------+-------+



Le CA est très stable, autour de 230 millions par an. 2005 est plus bas (219 millions) mais c'est parce que les ventes s'arrêtent le 13 décembre : `nb_mois` dit 12, mais décembre 2005 n'est pas complet.

In [6]:
# classement des magasins sur toute la période (Q5)
ventes_magasin_mois.groupBy("s_store_name").agg(
    F.sum("ca").alias("ca"),
    F.sum("nb_tickets").alias("nb_tickets"),
).withColumn("panier_moyen", F.round(F.col("ca") / F.col("nb_tickets"), 2)) \
 .orderBy(F.desc("ca")).show()

+------------+-----------+----------+------------+
|s_store_name|         ca|nb_tickets|panier_moyen|
+------------+-----------+----------+------------+
|     Buffalo|98950137.76|      7645|    12943.12|
|     Elmwood|96770105.59|      7462|    12968.39|
|  Germantown|96748356.68|      7435|    13012.56|
|      Forest|96727851.94|      7456|    12973.16|
|Williamstown|96578622.89|      7414|    13026.52|
|     Pioneer|96063990.59|      7396|    12988.64|
|    Hillside|95929844.66|      7422|    12925.07|
|     Bedford|95607843.48|      7449|    12834.99|
|  Bloomfield|94751173.08|      7284|    13008.12|
|      Avalon|94263977.55|      7409|    12722.90|
|       Egypt|94097279.37|      7369|    12769.34|
|    Sherwood|94040402.78|      7359|    12778.96|
+------------+-----------+----------+------------+



Les 12 magasins sont très proches (entre 94 et 99 millions sur 5 ans). Buffalo est premier, Sherwood dernier, mais l'écart n'est que de 5 %. Données synthétiques = ventes réparties presque au hasard.

## 3. Gold 2 – ventes par catégorie et par année

- **Granularité** : une ligne = une catégorie de produit pour une année.
- **Mesures** : CA, quantité vendue, nombre de lignes de vente, et la part de la catégorie dans le CA de l'année.

In [7]:
from pyspark.sql import Window

ventes_categorie_annee = (
    ventes_enrichies
    .groupBy("i_category", "d_year")
    .agg(
        F.sum("ss_net_paid").alias("ca"),
        F.sum("ss_quantity").alias("quantite"),
        F.count("*").alias("nb_lignes"),
    )
    .withColumnRenamed("i_category", "categorie")
    .withColumnRenamed("d_year", "annee")
)

total_annee = Window.partitionBy("annee")
ventes_categorie_annee = ventes_categorie_annee.withColumn(
    "part_ca_pct", F.round(F.col("ca") / F.sum("ca").over(total_annee) * 100, 2)
).orderBy("annee", F.desc("ca"))

ventes_categorie_annee.filter("annee = 2004").show()
print("nombre de lignes :", ventes_categorie_annee.count())

+--------------------+-----+------------+--------+---------+-----------+
|           categorie|annee|          ca|quantite|nb_lignes|part_ca_pct|
+--------------------+-----+------------+--------+---------+-----------+
|               Music| 2004|106452534.79| 3114031|    61358|      45.43|
|               Books| 2004| 62720328.90| 1838029|    36211|      26.76|
|         Movies & TV| 2004| 15029271.65|  446264|     8835|       6.41|
|Clothing & Access...| 2004| 11306354.16|  329126|     6548|       4.82|
|         Electronics| 2004| 11067717.87|  326515|     6512|       4.72|
|      Home & Kitchen| 2004|  8455390.61|  253791|     5053|       3.61|
|   Sports & Outdoors| 2004|  8411609.87|  250099|     4937|       3.59|
|        Toys & Games| 2004|  5475140.37|  163933|     3268|       2.34|
|Tools & Home Impr...| 2004|  5426586.07|  165298|     3292|       2.32|
+--------------------+-----+------------+--------+---------+-----------+



nombre de lignes : 45


45 lignes = 9 catégories × 5 ans. En 2004, Music fait 45 % du CA et Books 27 %. C'est logique : ce sont aussi les catégories qui ont le plus de produits dans `item` (8 143 et 5 335).

## 4. Écriture en Parquet et relecture

In [8]:
gold = {
    "ventes_magasin_mois": ventes_magasin_mois,
    "ventes_categorie_annee": ventes_categorie_annee,
}

for nom, df in gold.items():
    chemin = str(GOLD / nom)
    df.write.mode("overwrite").parquet(chemin)

    relu = spark.read.parquet(chemin)
    print(nom)
    print("  lignes écrites / relues :", df.count(), "/", relu.count())
    print("  mêmes types             :", relu.dtypes == df.dtypes)
    relu.printSchema()

ventes_magasin_mois


  lignes écrites / relues : 720 / 720
  mêmes types             : True
root
 |-- s_store_sk: long (nullable = true)
 |-- s_store_name: string (nullable = true)
 |-- s_state: string (nullable = true)
 |-- annee: integer (nullable = true)
 |-- mois: integer (nullable = true)
 |-- ca: decimal(17,2) (nullable = true)
 |-- quantite: long (nullable = true)
 |-- nb_tickets: long (nullable = true)
 |-- nb_lignes: long (nullable = true)
 |-- marge: decimal(17,2) (nullable = true)
 |-- panier_moyen: decimal(18,2) (nullable = true)



ventes_categorie_annee


  lignes écrites / relues : 45 / 45
  mêmes types             : True
root
 |-- categorie: string (nullable = true)
 |-- annee: integer (nullable = true)
 |-- ca: decimal(17,2) (nullable = true)
 |-- quantite: long (nullable = true)
 |-- nb_lignes: long (nullable = true)
 |-- part_ca_pct: decimal(24,2) (nullable = true)



## 5. Comparaison des formats

Je compare le fichier texte d'origine (`.dat`, qui est un CSV avec `|`) et le Parquet Silver, sur la même table `store_sales`.

In [9]:
def taille_mo(chemin):
    if chemin.is_file():
        return round(chemin.stat().st_size / 1024 / 1024, 1)
    return round(sum(f.stat().st_size for f in chemin.rglob("*.parquet")) / 1024 / 1024, 1)

for table, dossier in [("store_sales", "sales"), ("item", "products"), ("date_dim", "dates")]:
    print(f"{table:12} texte : {taille_mo(chemin_source(table)):>6} Mo   Parquet : {taille_mo(SILVER / dossier):>6} Mo")

store_sales  texte :   88.5 Mo   Parquet :   42.0 Mo
item         texte :    6.5 Mo   Parquet :    3.4 Mo
date_dim     texte :   14.7 Mo   Parquet :    3.5 Mo


In [10]:
# temps pour calculer le CA total à partir de chaque format
debut = time.time()
lire_brut(spark, "store_sales").select(F.sum(F.col("ss_net_paid").cast("decimal(7,2)"))).first()
print("texte   :", round(time.time() - debut, 2), "s")

debut = time.time()
spark.read.parquet(str(SILVER / "sales")).select(F.sum("ss_net_paid")).first()
print("Parquet :", round(time.time() - debut, 2), "s")

texte   : 1.15 s
Parquet : 0.2 s


Le temps varie d'une exécution à l'autre (c'est mon PC, pas un vrai test de performance), mais Parquet est plus rapide (4 à 5 fois plus rapide chez moi) parce qu'il ne lit que la colonne `ss_net_paid`, alors que le texte oblige à lire toutes les lignes en entier.

| | Texte délimité (`.dat` / CSV) | JSON | Parquet |
|---|---|---|---|
| Organisation | par lignes | par lignes (un objet par ligne) | par colonnes |
| Lisible à l'œil | oui | oui | non (binaire) |
| Types | aucun, tout est du texte | quelques types (nombre, texte) | types stockés dans le fichier |
| Volume observé (store_sales) | 88,5 Mo | pas utilisé dans le kit | 42 Mo (compressé) |
| Usage adapté | échange de fichiers, données brutes | API, données semi-structurées | analyse, agrégations sur quelques colonnes |
| Limites | lent à lire, types à redéfinir | encore plus gros que le CSV | illisible sans outil, pas fait pour modifier une ligne |

Attention : un dossier Parquet n'est pas une table transactionnelle. Si l'écriture plante au milieu, on peut avoir des fichiers incomplets, et il n'y a pas d'historique des versions. Il faudrait un format comme Delta Lake ou Iceberg pour ça (partie avancée du TP).

## Limites de mes données Gold

- 2005 est incomplète (ventes jusqu'au 13/12) : on ne peut pas comparer 2005 aux autres années telles quelles.
- Les données sont synthétiques (TPCx-BB) : les tendances ne représentent pas une vraie enseigne.
- La marge est négative pour la plupart des ventes à cause du générateur : je la garde dans le Gold mais elle ne doit pas être lue comme une vraie rentabilité.
- Seules les ventes magasin sont prises en compte, pas les ventes web ni les retours.